In [56]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.correlation_tools import cov_nearest
from scipy.optimize import minimize
import partieA as pa
from itertools import combinations
import time

In [57]:
data = pa.load_data('/Users/williambourque/Documents/HEC/H26 Gestion de Portefeuille/gestion_portefeuille_tp1/data/48_Industry_Portfolios.csv')
data_last_five = pa.extract_last_five_years(data, industries = list(data))

In [ ]:
def max_sharpe(returns, R=-5, annualize=True, long_only=False):
    sigma = pa.compute_sigma(returns)

    tf = 12 if annualize else 1
    mu = returns.mean().to_numpy() * tf
    Sigma = sigma.to_numpy() * tf
    ones = np.ones(len(mu))

    if long_only:
        w, _, _, sharpe = pa.tangency_portfolio_noshort(mu, Sigma, R=R)
        return w, sharpe

    invS = np.linalg.pinv(Sigma)
    excess = mu - R * ones
    z = invS @ excess
    w = z / (ones @ z)
    sharpe = float(np.sqrt(excess @ invS @ excess))
    return w, sharpe

In [59]:
# ---------- Brute force selection ----------

def best_5_industries_max_sharpe(df, R, annualize=True, long_only=False):
    # keep only columns with positive (annualized) mean
    tf = 12 if annualize else 1
    cols = df.columns[(df.mean() * tf) > 0]

    best_combo, best_w, best_s = None, None, -np.inf

    for combo in combinations(cols, 5):
        sub = df[list(combo)].dropna()
        if long_only:
            w, s = max_sharpe(sub, R=R, annualize=annualize)

        if s > best_s:
            best_combo, best_w, best_s = combo, w, s

    weights = pd.Series(best_w, index=list(best_combo))
    return best_combo, weights, best_s

# t_brute_0 = time.perf_counter()
# best_combo_ls, weights_ls, best_s_ls = best_5_industries_max_sharpe(data_last_five, annualize=True, long_only=False)
# best_combo_long, weights_long, best_s_long = best_5_industries_max_sharpe(data_last_five, annualize=True, long_only=True)
#t_brute_1 = time.perf_counter()

# print(best_s_ls)
# print(best_s_long)
# print(weights_ls)
# print(weights_long)

In [60]:
# ---------- heuristic selection ----------

def best_5_industries_greedy(
    df, R, annualize=True, long_only=False
):
    tf = 12 if annualize else 1
    cols = df.columns[(df.mean() * tf) > 0].tolist()
    if len(cols) < 5:
        raise ValueError("Not enough positive-mean industries to pick 5.")

    data = df[cols].dropna()

    selected = []
    remaining = cols[:]

    while len(selected) < 5:
        best_cand, best_s = None, -np.inf
        for c in remaining:
            trial = selected + [c]
            _, s = max_sharpe(data[trial], R=R, annualize=annualize, long_only=long_only)
            if s > best_s:
                best_s, best_cand = s, c
        selected.append(best_cand)
        remaining.remove(best_cand)


    # final weights + sharpe
    w, s = max_sharpe(data[selected], R=R, annualize=annualize, long_only=long_only)
    weights = pd.Series(w, index=selected)
    return tuple(selected), weights, s

In [61]:
t_heuristic_0 = time.perf_counter()
sel_ls, w_ls, s_ls = best_5_industries_greedy(data_last_five, R=2, long_only=False)
sel_long, w_long, s_long = best_5_industries_greedy(data_last_five, R=2, long_only=True)
t_heuristic_1 = time.perf_counter()

print(f"Heuristic approach took {t_heuristic_1 - t_heuristic_0:.4f} seconds")
print(w_ls)
print(s_ls)
print(w_long)
print(s_long)

Heuristic approach took 0.2301 seconds
Coal     1.288926
Paper   -2.814370
Steel    2.571230
RlEst   -4.377231
Fin      4.331446
dtype: float64
2.6611864924159243
Coal     0.380237
Util     0.323924
FabPr    0.079694
Soda     0.095629
Insur    0.120515
dtype: float64
1.5979441031473043
